# Data cleaning — Demographic Data.xlsx

Builds the four CSVs used by **main.py**: `data/age.csv`, `data/race.csv`, `data/occupation.csv`, `data/income.csv` (Alameda County). Run this notebook when the source Excel changes; otherwise use main.py for agent generation.

In [9]:
import os
import re
import pandas as pd

ROOT = os.getcwd()
XLSX_PATH = os.path.join(ROOT, "Demographic Data.xlsx")
assert os.path.isfile(XLSX_PATH), f"Not found: {XLSX_PATH}"

def parse_value(raw):
    if pd.isna(raw) or raw in ("(X)", "*****", ""):
        return None
    s = str(raw).strip().replace(",", "")
    if s.endswith("%"):
        try: return float(s[:-1])
        except ValueError: return None
    s = re.sub(r"^±\s*", "", s)
    try: return int(float(s))
    except ValueError: return None

# Age: category, count (rows 5-22)
df_age_raw = pd.read_excel(XLSX_PATH, sheet_name="Age", header=None)
df_age = pd.DataFrame([{"category": str(df_age_raw.iloc[i, 0]).strip(), "count": parse_value(df_age_raw.iloc[i, 1])}
    for i in range(5, 23) if not pd.isna(df_age_raw.iloc[i, 0]) and parse_value(df_age_raw.iloc[i, 1]) is not None])

# Race: category, value (6 under Population of one race, rows 2-7)
df_race_raw = pd.read_excel(XLSX_PATH, sheet_name="Race")
df_race = pd.DataFrame([{"category": str(df_race_raw.iloc[i, 0]).strip(), "value": parse_value(df_race_raw.iloc[i, 1])}
    for i in range(2, 8)])

# Occupation: category, totalestimate (from row 3)
df_occ_raw = pd.read_excel(XLSX_PATH, sheet_name="Occupation", header=None)
df_occupation = pd.DataFrame([{"category": str(df_occ_raw.iloc[i, 0]).strip(), "totalestimate": parse_value(df_occ_raw.iloc[i, 1])}
    for i in range(3, len(df_occ_raw)) if not pd.isna(df_occ_raw.iloc[i, 0]) and parse_value(df_occ_raw.iloc[i, 1]) is not None])

# Income: category, estimated (household brackets rows 4-13)
df_inc_raw = pd.read_excel(XLSX_PATH, sheet_name="Income", header=None)
df_income = pd.DataFrame([{"category": str(df_inc_raw.iloc[i, 0]).strip(), "estimated": parse_value(df_inc_raw.iloc[i, 1])}
    for i in range(4, 14) if not pd.isna(df_inc_raw.iloc[i, 0]) and parse_value(df_inc_raw.iloc[i, 1]) is not None])

OUT_DIR = os.path.join(ROOT, "data")
os.makedirs(OUT_DIR, exist_ok=True)
df_age.to_csv(os.path.join(OUT_DIR, "age.csv"), index=False)
df_race.to_csv(os.path.join(OUT_DIR, "race.csv"), index=False)
df_occupation.to_csv(os.path.join(OUT_DIR, "occupation.csv"), index=False)
df_income.to_csv(os.path.join(OUT_DIR, "income.csv"), index=False)
print("Saved: data/age.csv, data/race.csv, data/occupation.csv, data/income.csv")

Saved: data/age.csv, data/race.csv, data/occupation.csv, data/income.csv
